# Q7 - Code Review_Required Capstone_12.1

In [7]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ExpSineSquared, DotProduct
from sklearn.gaussian_process import GaussianProcessClassifier, GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic, WhiteKernel, ConstantKernel
from scipy.stats import norm
from skopt import gp_minimize
from skopt.space import Real
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import differential_evolution
from scipy.optimize import minimize

In [8]:
inputs7 = np.array([[0.27262382, 0.32449536, 0.89710881, 0.83295115, 0.15406269,
        0.79586362],
       [0.54300258, 0.9246939 , 0.34156746, 0.64648585, 0.71844033,
        0.34313266],
       [0.09083225, 0.66152938, 0.06593091, 0.25857701, 0.96345285,
        0.6402654 ],
       [0.11886697, 0.61505494, 0.90581639, 0.8553003 , 0.41363143,
        0.58523563],
       [0.63021764, 0.8380969 , 0.68001305, 0.73189509, 0.52673671,
        0.34842921],
       [0.76491917, 0.25588292, 0.60908422, 0.21807904, 0.32294277,
        0.09579366],
       [0.05789554, 0.49167222, 0.24742222, 0.21811844, 0.42042833,
        0.73096984],
       [0.19525188, 0.07922665, 0.55458046, 0.17056682, 0.01494418,
        0.10703171],
       [0.64230298, 0.83687455, 0.02179269, 0.10148801, 0.68307083,
        0.6924164 ],
       [0.78994255, 0.19554501, 0.57562333, 0.07365919, 0.25904917,
        0.05109986],
       [0.52849733, 0.45742436, 0.36009569, 0.36204551, 0.81689098,
        0.63747637],
       [0.72261522, 0.01181284, 0.06364591, 0.16517311, 0.07924415,
        0.35995166],
       [0.07566492, 0.33450212, 0.13273274, 0.60831236, 0.91838592,
        0.82233079],
       [0.94245084, 0.37743962, 0.48612233, 0.22879108, 0.08263175,
        0.71195755],
       [0.14864702, 0.03394336, 0.72880565, 0.31606646, 0.02176938,
        0.51691776],
       [0.81711239, 0.54816823, 0.10334758, 0.12436955, 0.72823482,
        0.44967361],
       [0.41762629, 0.06409998, 0.24566877, 0.5590408 , 0.19153138,
        0.25464092],
       [0.72628566, 0.46489581, 0.92457051, 0.8072454 , 0.6354384 ,
        0.14341787],
       [0.31981043, 0.52009759, 0.29067775, 0.87670668, 0.49503469,
        0.6190825 ],
       [0.87987128, 0.39796199, 0.00363456, 0.95699064, 0.26451373,
        0.11486924],
       [0.54124078, 0.63140314, 0.03190205, 0.44998156, 0.79865282,
        0.63370429],
       [0.22634792, 0.11502581, 0.82474966, 0.94538372, 0.90531153,
        0.95101392],
       [0.68685257, 0.04101721, 0.00757301, 0.285009  , 0.69156848,
        0.6555429 ],
       [0.17597754, 0.6244165 , 0.29554198, 0.46955276, 0.09776977,
        0.72814108],
       [0.88164674, 0.20445019, 0.41447436, 0.42038468, 0.26491501,
        0.73066019],
       [0.06661051, 0.52804507, 0.8160952 , 0.96101714, 0.08650933,
        0.77778822],
       [0.93246638, 0.48881189, 0.25860774, 0.95624344, 0.19042781,
        0.51985176],
       [0.84686697, 0.14242917, 0.06066859, 0.75629213, 0.5523983 ,
        0.08130609],
       [0.80628208, 0.32412237, 0.72607601, 0.14871213, 0.7193764 ,
        0.36288398],
       [0.47682313, 0.34094195, 0.01433523, 0.88013956, 0.9986547 ,
        0.07966402], 
       [0.066287, 0.429544, 0.217458, 0.229218, 0.311516, 0.787145],
        [0.008389, 0.059503, 0.568726, 0.035273, 0.026110, 0.996763],
        [0.061207, 0.392516, 0.202916, 0.216277, 0.280103, 0.811126],
        [0.009524, 0.381319, 0.130135, 0.173740, 0.280292, 0.824702]])

In [9]:
outputs7 = np.array([0.6044327 , 0.56275307, 0.00750324, 0.0614243 , 0.2730468 ,
       0.08374657, 1.3649683 , 0.09264495, 0.0178696 , 0.03356494,
       0.0735163 , 0.2063097 , 0.00882563, 0.26840032, 0.61152553,
       0.01479818, 0.27489251, 0.06676325, 0.04211835, 0.00270147,
       0.01820907, 0.00701603, 0.10050661, 0.47539552, 0.67514163,
       0.51645722, 0.00377748, 0.00313433, 0.02134252, 0.09541116, 1.767593736909062, 
       0.3088683369137646,1.7219502816061625,1.3917502403938542])

## Surrogate function

In [12]:
X = inputs7
y = outputs7

# Define a robust kernel
kernel_best_Q7 = RBF(length_scale=0.5, length_scale_bounds=(1e-3, 1e2)) + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-9, 1e-1))

gp = GaussianProcessRegressor(
    kernel=kernel_best_Q7,
    n_restarts_optimizer=10,
    alpha=1e-6,
    normalize_y=True)

# Fit GP
gp.fit(X, y)
print("Optimized kernel:", gp.kernel_)


Optimized kernel: RBF(length_scale=0.237) + WhiteKernel(noise_level=6.09e-09)


## Acquisition function - Optimization (UCB - global-local optimizer)

In [13]:
def hybrid_ucb(X, gp, kappa=0.5, alpha=0.5):
    mu, sigma = gp.predict(X, return_std=True)
    return alpha * mu + (1 - alpha) * (mu + kappa * sigma)


# ----------------------------------------------------
# 2. Global + local optimizer for the acquisition
# ----------------------------------------------------
def optimize_acquisition_global(acquisition_func, gp, bounds, n_random=5000, n_local=10, **kwargs):
    dim = len(bounds)

    # Step 1: Random exploration
    X_random = np.random.uniform(
        [b[0] for b in bounds],
        [b[1] for b in bounds],
        size=(n_random, dim)
    )
    y_random = acquisition_func(X_random, gp, **kwargs)

    # Step 2: Choose top n_local points for local refinement
    top_idx = np.argsort(y_random)[-n_local:]
    best_val = -np.inf
    best_x = None

    # Step 3: Local optimization (L-BFGS-B) from multiple starts
    for idx in top_idx:
        x0 = X_random[idx]

        def objective(x):
            return -acquisition_func(x.reshape(1, -1), gp, **kwargs)

        res = minimize(objective, x0=x0, bounds=bounds, method="L-BFGS-B")
        if not res.success:
            continue

        val = -res.fun
        if val > best_val:
            best_val = val
            best_x = res.x

    return best_x

In [17]:
bounds = [(0, 1), (0, 1), (0, 1), (0 ,1), (0, 1), (0 , 1)]

# Optimize acquisition
next_point = optimize_acquisition_global(hybrid_ucb, gp, bounds, n_random=10000, n_local=5000, kappa=1, alpha=0.5)
#higher alpha

print("Next query point (by UCB):", next_point)

Next query point (by UCB): [0.10924322 0.42295653 0.24521378 0.2663769  0.2944423  0.81192075]


## Acquisition function - Bayesian Optimization (UCB - Pure DE)

In [16]:
def optimize_acquisition_global_de(acquisition_func, gp, bounds, kappa=4.0, maxiter=100, popsize=30):
    # Objective to minimize (negative because DE minimizes)
    def objective_de(x):
        x = np.array(x).reshape(1, -1)
        return -acquisition_func(x, gp, kappa=kappa)[0]

    # Differential Evolution global optimization
    result = differential_evolution(
        objective_de,
        bounds=bounds,
        maxiter=maxiter,
        popsize=popsize,
        polish=True,       # refine the result locally at the end
        disp=True           # print optimization progress
    )

    x_next = result.x
    best_val = -result.fun
    return x_next, best_val

In [19]:
bounds = [(0, 1), (0, 1), (0, 1), (0 ,1), (0, 1), (0 , 1)]

next_point1, best_val1 = optimize_acquisition_global_de(
        acquisition_func=hybrid_ucb,
        gp=gp,
        bounds=bounds,
        kappa=1,      # larger kappa → more exploration
        maxiter=20000,     # number of DE iterations
        popsize=5000      # population size
    )

# Optimize acquisition
print("Next query point (by UCB):", next_point1)

differential_evolution step 1: f(x)= -1.7570272191430003
differential_evolution step 2: f(x)= -1.7570272191430003
differential_evolution step 3: f(x)= -1.8086321326096286
differential_evolution step 4: f(x)= -1.8086321326096286
differential_evolution step 5: f(x)= -1.8125362777208651
differential_evolution step 6: f(x)= -1.8295933837575347
differential_evolution step 7: f(x)= -1.834025387486561
differential_evolution step 8: f(x)= -1.834025387486561
differential_evolution step 9: f(x)= -1.834025387486561
differential_evolution step 10: f(x)= -1.8387179042937172
differential_evolution step 11: f(x)= -1.8387179042937172
differential_evolution step 12: f(x)= -1.839913718869496
differential_evolution step 13: f(x)= -1.839913718869496
differential_evolution step 14: f(x)= -1.839913718869496
differential_evolution step 15: f(x)= -1.839913718869496
differential_evolution step 16: f(x)= -1.8400469755463518
Polishing solution with 'L-BFGS-B'
Next query point (by UCB): [0.10921962 0.42296567 0.2